# Phase 7 v4 · Legal passage retrieval and learned reranking

Attach **phase2v4**, enable Kaggle Internet and the HF secret, and enable a GPU. GPU_INDEX = 0 selects the first device. The first code cell defaults to **SMOKE_MODE = True**. This exercises the real checkpoint on small, reproducible subsets and writes smoke_prediction.json. Set it to False and Run All for the competition file.

The base is final_model(2).pt from the latest Phase 2 v4 notebook, with its original model code and tokenizer. The checkpoint config supplies the exact layer count and width. The worker strictly loads the weights. Local processing has a **70-minute cutoff**. Hugging Face upload is optional and separate.

## What changed and why

- V3's reranker scored one short passage pair, reweighted it heavily, and could only reorder a small shortlist. V4 keeps full-document lexical scores, passage-level lexical scores, and several neural passage matches. Statute and section overlaps use provided text.
- Start LoRMA+ from the Phase 2 v4 base, using about 65% of the eligible Train queries. Use document and passage BM25 negatives, then mine dense negatives once after the first epoch. The remaining Train query groups are held out from neural updates and used to train LambdaMART.
- Full-mode neural training uses no epoch cap. Three consecutive Dev loss checks without improvement of at least 0.001 trigger early stopping. Select the checkpoint by Dev Micro-F1@7, using pairwise Dev loss to break ties. Smoke mode limits training to two small passes.
- Combine the top 60 document BM25, passage BM25, and dense candidates, plus the hybrid baseline's top seven. Log shortlist recall and the oracle Micro-F1@7 ceiling. LambdaMART learns to reorder candidates using neural, lexical, statute, and rank features. Use the learned ranker for the final submission only if it beats the hybrid baseline on Dev.
- A 0.45 score is a target. Dev ceilings and scores measure progress on the supplied labels.

Sources

- [Althammer et al. · Paragraph aggregation for legal retrieval](https://arxiv.org/abs/2201.01614)
- [Xiong et al. · Dense hard-negative mining](https://arxiv.org/abs/2007.00808)
- [Burges · LambdaMART](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/MSR-TR-2010-82.pdf)
- [LightGBM ranking API](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRanker.html)

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess
import sys
import time

SMOKE_MODE = False  # Switch to False for a competition run.
STARTED = time.monotonic()
DEADLINE = STARTED + 70 * 60
GPU_INDEX = 0
CHECKPOINT = Path("/kaggle/input/datasets/koooshiieee/phase2v4/final_model(2).pt")
LOCAL_DATA_ROOT = None  # Folder containing Train/, Dev/, and Test/ if attached.
LOCAL_TOKENIZER_DIR = None  # Folder with the original vocab.json and merges.json if attached.
CHUNK_LENGTH = 128 if SMOKE_MODE else 384
WINDOW_STRIDE = 96 if SMOKE_MODE else 320
MAX_WINDOWS = 3 if SMOKE_MODE else 10
SHORTLIST_DEPTH = 20 if SMOKE_MODE else 60
QUERY_BATCH = 2
ENCODE_BATCH = 8
VALIDATION_SECONDS = 180
prefix = "smoke" if SMOKE_MODE else "full"
RUN_ROOT = Path("/kaggle/working/phase7_v4") / (
    prefix + "_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
)
RUN_ROOT.mkdir(parents=True)


## Materialize the original model, tokenizer, and retrieval workers

In [ ]:
source = r'''"""
Optimized Modern Legal LLM (from scratch):
- Pre-norm RMSNorm
- QK-Norm (RMSNorm on Query and Key vectors before attention)
- Rotary Position Embeddings (RoPE)
- Grouped Query Attention (GQA) with fused SDPA
- Parallel Attention & SwiGLU FFN (PaLM / GPT-J style)
- Deep & Narrow parameter presets (MobileLLM efficiency)
- Tied input/output embeddings
"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------------------------------------------------------- Config
@dataclass
class Config:
    vocab_size: int = 33000
    d_model: int = 672
    n_layers: int = 18
    n_heads: int = 14
    n_kv_heads: int = 2          # 14 query heads share 2 KV heads (7:1 GQA)
    ffn_hidden: int = 1792
    max_seq_len: int = 1280
    rope_cache_len: int = 4096
    rope_theta: float = 10000.0
    norm_eps: float = 1e-5
    tie_embeddings: bool = True
    pos_enc: str = "rope"        # "rope"
    attn_type: str = "gqa"       # "gqa"
    attn_impl: str = "sdpa"      # "sdpa" (PyTorch fused kernel)
    use_qk_norm: bool = True     # QK-Norm for training stability
    parallel_attn: bool = True   # Parallel Attention + FFN

PRESETS = {
    "deep_narrow_120m": dict(d_model=640, n_layers=20, n_heads=10, n_kv_heads=2, ffn_hidden=1664),
    "arch_deep_narrow_135m": dict(d_model=672, n_layers=18, n_heads=14, n_kv_heads=2, ffn_hidden=1792),
    "tiny": dict(d_model=64, n_layers=2, n_heads=4, n_kv_heads=2, ffn_hidden=128),
}

def count_params(model, verbose=True) -> int:
    total = sum(p.numel() for p in model.parameters())
    if verbose:
        emb = model.tok_emb.weight.numel()
        blocks = sum(p.numel() for p in model.layers.parameters())
        print(f"Total: {total/1e6:.2f}M | Transformer Blocks: {blocks/1e6:.2f}M | Embeddings: {emb/1e6:.2f}M")
    return total

# ----------------------------------------------------------------------------- RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

# ----------------------------------------------------------------------------- RoPE
def build_rope_cache(head_dim, max_len, theta):
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_len).float()
    freqs = torch.outer(t, inv_freq)
    return freqs.cos(), freqs.sin()

def apply_rope(x, cos, sin):
    half = x.shape[-1] // 2
    xf = x.float()
    x1, x2 = xf[..., :half], xf[..., half:]
    cos = cos[None, None]
    sin = sin[None, None]
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)

# ----------------------------------------------------------------------------- GQA Attention with QK-Norm
def repeat_kv(x, n_rep):
    if n_rep == 1:
        return x
    B, n_kv, S, hd = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, S, hd).reshape(B, n_kv * n_rep, S, hd)

class GQAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0 and cfg.n_heads % cfg.n_kv_heads == 0
        self.nh = cfg.n_heads
        self.nkv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.n_rep = self.nh // self.nkv
        self.impl = cfg.attn_impl
        self.use_qk_norm = cfg.use_qk_norm

        self.wq = nn.Linear(cfg.d_model, self.nh * self.hd, bias=False)
        self.wk = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(cfg.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, cfg.d_model, bias=False)

        if self.use_qk_norm:
            self.q_norm = RMSNorm(self.hd, cfg.norm_eps)
            self.k_norm = RMSNorm(self.hd, cfg.norm_eps)

    def forward(self, x, cos, sin, key_valid=None, past=None, use_cache=False):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)

        # Apply QK-Norm
        if self.use_qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        # Apply RoPE
        if cos is not None:
            q = apply_rope(q, cos, sin)
            k = apply_rope(k, cos, sin)

        if past is not None:
            k = torch.cat([past[0], k], dim=2)
            v = torch.cat([past[1], v], dim=2)
        new_cache = (k, v) if use_cache else None
        S = k.shape[2]

        k = repeat_kv(k, self.n_rep)
        v = repeat_kv(v, self.n_rep)

        if self.impl == "sdpa" and key_valid is None and T == S:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:
            qpos = torch.arange(S - T, S, device=x.device)
            kpos = torch.arange(S, device=x.device)
            rel = qpos[:, None] - kpos[None, :]
            allowed = (rel >= 0)[None, None]
            if key_valid is not None:
                allowed = allowed & key_valid[:, None, None, :]
            scores = ((q * (self.hd ** -0.5)) @ k.transpose(-2, -1)).float()
            scores = scores.masked_fill(~allowed, -1e30)
            probs = torch.softmax(scores, dim=-1).to(v.dtype)
            out = probs @ v

        out = out.transpose(1, 2).reshape(B, T, self.nh * self.hd)
        return self.wo(out), new_cache

# ----------------------------------------------------------------------------- SwiGLU FFN
class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.w_gate = nn.Linear(d_model, hidden, bias=False)
        self.w_up = nn.Linear(d_model, hidden, bias=False)
        self.w_down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

# ----------------------------------------------------------------------------- Transformer Block
class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.parallel_attn = cfg.parallel_attn
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = GQAttention(cfg)
        self.mlp_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mlp = SwiGLU(cfg.d_model, cfg.ffn_hidden)

    def forward(self, x, cos, sin, key_valid=None, past=None, use_cache=False):
        if self.parallel_attn:
            # Parallel Attention & FFN (PaLM style)
            h_attn, new_cache = self.attn(self.attn_norm(x), cos, sin, key_valid, past, use_cache)
            h_mlp = self.mlp(self.mlp_norm(x))
            x = x + h_attn + h_mlp
        else:
            h, new_cache = self.attn(self.attn_norm(x), cos, sin, key_valid, past, use_cache)
            x = x + h
            x = x + self.mlp(self.mlp_norm(x))
        return x, new_cache

# ----------------------------------------------------------------------------- Full LM
class LlamaLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.layers = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight

        cos, sin = build_rope_cache(cfg.d_model // cfg.n_heads, cfg.rope_cache_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("wo.weight") or name.endswith("w_down.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, use_cache=False, cache=None, start_pos=0,
                key_valid=None, all_logits=False):
        B, T = idx.shape
        x = self.tok_emb(idx)
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]

        new_caches = []
        for i, layer in enumerate(self.layers):
            past = cache[i] if cache is not None else None
            x, c = layer(x, cos, sin, key_valid=key_valid, past=past, use_cache=use_cache)
            new_caches.append(c)
        x = self.norm(x)

        if targets is not None:
            logits = self.lm_head(x)
            return F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        logits = self.lm_head(x if all_logits else x[:, -1:, :])
        return logits, (new_caches if use_cache else None)

# ----------------------------------------------------------------------------- Generation
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id, pad_id, temperature=0.2, top_k=0,
             top_p=0.9, rep_penalty=1.12, ban_eos=False):
    model.eval()
    device = next(model.parameters()).device
    V = model.cfg.vocab_size
    B = len(prompts)
    L = max(len(p) for p in prompts)
    ids = torch.full((B, L), pad_id, dtype=torch.long, device=device)
    key_valid = torch.zeros(B, L, dtype=torch.bool, device=device)
    seen = torch.zeros(B, V, dtype=torch.bool, device=device)

    for b, p in enumerate(prompts):
        t = torch.tensor(p, dtype=torch.long, device=device)
        ids[b, L - len(p):] = t
        key_valid[b, L - len(p):] = True
        seen[b, t] = True

    out = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    amp = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda"))

    with amp:
        logits, cache = model(ids, use_cache=True, key_valid=key_valid, start_pos=0)
        for step in range(max_new_tokens):
            lg = logits[:, -1, :].float()
            if ban_eos:
                lg[:, eos_id] = float("-inf")
            if rep_penalty != 1.0:
                pen = torch.where(lg > 0, lg / rep_penalty, lg * rep_penalty)
                lg = torch.where(seen, pen, lg)

            if temperature <= 0.05:
                nxt = lg.argmax(-1)
            else:
                lg = lg / temperature
                if top_k and top_k > 0:
                    kth = torch.topk(lg, min(top_k, V)).values[:, -1:]
                    lg = lg.masked_fill(lg < kth, float("-inf"))
                if top_p < 1.0:
                    sl, si = torch.sort(lg, descending=True, dim=-1)
                    pr = torch.softmax(sl, dim=-1)
                    drop = (pr.cumsum(-1) - pr) > top_p
                    sl = sl.masked_fill(drop, float("-inf"))
                    lg = torch.full_like(lg, float("-inf")).scatter(1, si, sl)
                nxt = torch.multinomial(torch.softmax(lg, dim=-1), 1).squeeze(1)

            nxt = torch.where(finished, torch.full_like(nxt, pad_id), nxt)
            for b in range(B):
                if not finished[b] and nxt[b].item() != eos_id:
                    out[b].append(int(nxt[b].item()))
            finished = finished | (nxt == eos_id)
            seen.scatter_(1, nxt[:, None], True)
            if bool(finished.all()):
                break
            key_valid = torch.cat([key_valid, torch.ones(B, 1, dtype=torch.bool, device=device)], dim=1)
            logits, cache = model(nxt[:, None], use_cache=True, cache=cache,
                                  start_pos=L + step, key_valid=key_valid)
    return out
'''
(RUN_ROOT / 'llama_scratch.py').write_text(source, encoding='utf-8')
print('Prepared llama_scratch.py')

In [ ]:
source = r'''#!/usr/bin/env python3
import os, json, regex
from typing import Dict, List, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import regex


_PAT = regex.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")


class PureByteLevelBPETokenizer:
    def __init__(self, vocab_size: int = 33000):
        self.vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>", "<mask>"]

        self.encoder: Dict[bytes, int] = {}
        self.decoder: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], bytes] = {}       # pair -> merged token
        self.merge_ranks: Dict[Tuple[bytes, bytes], int] = {}    # pair -> order learned (lower = earlier)
        self._cache: Dict[str, List[int]] = {}

        self._init_base_vocab()

    def _init_base_vocab(self):
        for b in range(256):
            b_bytes = bytes([b])
            self.encoder[b_bytes] = b
            self.decoder[b] = b_bytes

        # special tokens
        for idx, token in enumerate(self.special_tokens, start=256):
            b_bytes = token.encode("utf-8")
            self.encoder[b_bytes] = idx
            self.decoder[idx] = b_bytes

    @property
    def pad_token_id(self) -> int:
        return self.encoder["<pad>".encode("utf-8")]

    @property
    def unk_token_id(self) -> int:
        return self.encoder["<unk>".encode("utf-8")]

    @property
    def bos_token_id(self) -> int:
        return self.encoder["<s>".encode("utf-8")]

    @property
    def eos_token_id(self) -> int:
        return self.encoder["</s>".encode("utf-8")]

    # ------------------------------------------------------------------ training
    @staticmethod
    def _merge_word(word, a, b, new):
        out, i, n = [], 0, len(word)
        while i < n:
            if i < n - 1 and word[i] == a and word[i + 1] == b:
                out.append(new)
                i += 2
            else:
                out.append(word[i])
                i += 1
        return out

    def train_from_texts(self, texts: List[str], target_vocab_size: int = None, min_word_freq: int = 2):
        if target_vocab_size is None:
            target_vocab_size = self.vocab_size
        num_merges = target_vocab_size - len(self.encoder)
        print(f"\nTraining byte level BPE from scratch. Target merges: {num_merges:,} | Base Vocab: {len(self.encoder)}")

        # 1) pre-tokenize + count unique words (the BPE then runs on this small table, not the raw text)
        word_counts = Counter()
        for text in tqdm(texts, desc="Counting pre-tokens"):
            if text and text.strip():
                word_counts.update(_PAT.findall(text))

        words, freqs = [], []
        for w, c in word_counts.items():
            if c >= min_word_freq:
                words.append([bytes([b]) for b in w.encode("utf-8", "replace")])
                freqs.append(c)
        del word_counts
        print(f"Unique words used for merge learning: {len(words):,}")

        # 2) initial pair statistics (updated incrementally afterwards)
        pair_counts = defaultdict(int)
        where = defaultdict(set)
        for i, w in enumerate(tqdm(words, desc="Initial pair counts")):
            for p in zip(w, w[1:]):
                pair_counts[p] += freqs[i]
                where[p].add(i)

        # 3) merge loop
        pbar = tqdm(total=num_merges, desc="Learning BPE Merges", dynamic_ncols=True)
        for _ in range(num_merges):
            if not pair_counts:
                print("No more candidate pairs found to merge.")
                break

            best_pair, freq = max(pair_counts.items(), key=lambda kv: kv[1])
            if freq < 2:
                print(f"Stopping early: Most frequent pair frequency is {freq}")
                break

            a, b = best_pair
            merged_token = a + b
            self.merges[best_pair] = merged_token
            self.merge_ranks[best_pair] = len(self.merge_ranks)
            if merged_token not in self.encoder:          # two merge paths can give the same bytes
                new_id = len(self.encoder)
                self.encoder[merged_token] = new_id
                self.decoder[new_id] = merged_token

            for i in list(where[best_pair]):
                w, f = words[i], freqs[i]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0:
                        del pair_counts[p]
                w = self._merge_word(w, a, b, merged_token)
                words[i] = w
                for p in zip(w, w[1:]):
                    pair_counts[p] += f
                    where[p].add(i)
            del where[best_pair]
            pair_counts.pop(best_pair, None)

            pbar.update(1)
            pbar.set_postfix({"vocab_size": len(self.encoder), "top_pair_freq": freq})

        pbar.close()
        self._cache.clear()
        print(f"Training finished! Final Vocab Size: {len(self.encoder):,}")

    # ------------------------------------------------------------------ encode / decode
    def _encode_word(self, word: str) -> List[int]:
        cached = self._cache.get(word)
        if cached is not None:
            return cached
        tokens = [bytes([b]) for b in word.encode("utf-8", "replace")]
        while len(tokens) >= 2:
            best_pair, best_rank = None, None
            for p in zip(tokens, tokens[1:]):
                r = self.merge_ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_pair, best_rank = p, r
            if best_pair is None:
                break
            tokens = self._merge_word(tokens, best_pair[0], best_pair[1], self.merges[best_pair])
        ids = [self.encoder.get(t, self.unk_token_id) for t in tokens]
        if len(self._cache) < 1_000_000:
            self._cache[word] = ids
        return ids

    def encode(self, text: str) -> List[int]:
        ids: List[int] = []
        for word in _PAT.findall(text):
            ids.extend(self._encode_word(word))
        return ids

    def decode(self, token_ids: List[int]) -> str:
        raw_bytes = b"".join(self.decoder.get(i, b"<unk>") for i in token_ids)
        return raw_bytes.decode("utf-8", errors="replace")

    # ------------------------------------------------------------------ save / load
    def save(self, save_dir: str):
        os.makedirs(save_dir, exist_ok=True)
        vocab_path = os.path.join(save_dir, "vocab.json")
        merges_path = os.path.join(save_dir, "merges.json")
        config_path = os.path.join(save_dir, "tokenizer_config.json")

        # latin1 maps bytes 1:1 onto code points, so any byte string survives JSON
        ser_vocab = {k.decode("latin1"): v for k, v in self.encoder.items()}
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(ser_vocab, f, ensure_ascii=False, indent=2)

        # JSON (not a space-separated text file): tokens can contain spaces, newlines, \r, \xa0 ...
        ser_merges = [[p1.decode("latin1"), p2.decode("latin1")] for (p1, p2) in self.merges]  # in learned order
        with open(merges_path, "w", encoding="utf-8") as f:
            json.dump(ser_merges, f, ensure_ascii=False)

        config = {
            "tokenizer_class": "PureByteLevelBPETokenizer",
            "vocab_size": len(self.encoder),
            "special_tokens": self.special_tokens,
            "pad_token_id": self.pad_token_id,
            "bos_token_id": self.bos_token_id,
            "eos_token_id": self.eos_token_id
        }
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2)

        print(f"Tokenizer saved successfully to {save_dir}")

    @classmethod
    def load(cls, save_dir: str) -> "PureByteLevelBPETokenizer":
        tok = cls()
        with open(os.path.join(save_dir, "vocab.json"), "r", encoding="utf-8") as f:
            ser_vocab = json.load(f)
        tok.encoder = {k.encode("latin1"): v for k, v in ser_vocab.items()}
        tok.decoder = {v: k.encode("latin1") for k, v in ser_vocab.items()}

        tok.merges, tok.merge_ranks = {}, {}
        with open(os.path.join(save_dir, "merges.json"), "r", encoding="utf-8") as f:
            for rank, (s1, s2) in enumerate(json.load(f)):
                p1, p2 = s1.encode("latin1"), s2.encode("latin1")
                tok.merges[(p1, p2)] = p1 + p2
                tok.merge_ranks[(p1, p2)] = rank
        return tok
'''
(RUN_ROOT / 'bpe_tokenizer.py').write_text(source, encoding='utf-8')
print('Prepared bpe_tokenizer.py')

In [ ]:
source = r'''from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path
from time import perf_counter
import json
import math
import os
import random
import re
import shutil
import subprocess

started = perf_counter()
DATA_REPO = "Exploration-Lab/CS-781-Capstone"
DATA_REVISION = "main"  # Set to a commit hash for repeatable inputs.
LOCAL_DATA_ROOT = None  # Directory containing Train/, Dev/, and Test/.
CHECKPOINT = Path("/kaggle/input/datasets/koooshiieee/phase2v1/final_model.pt")
TOKENIZER_REPO = "DumbLad/legal-bpe-tokenizer-phase1-v2"
LOCAL_TOKENIZER_DIR = None  # Directory containing the original vocab.json and merges.json.
TOKEN_SAMPLE_PER_POOL = 32  # Set to 0 to skip tokenizer sampling.
BM25_DEV_QUERIES = 80  # Set to 0 to skip, or None to score all Dev queries.
SEED = 781
SPLITS = {"Train": "train", "Dev": "dev", "Test": "test"}

def hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
    except Exception:
        return None
    for name in ("HF", "HF_TOKEN"):
        try:
            token = secrets.get_secret(name)
        except Exception:
            continue
        if token:
            return token
    return None

def hub_file(repo, filename, repo_type, revision=None):
    try:
        from huggingface_hub import hf_hub_download
        from huggingface_hub.errors import GatedRepoError
    except ImportError as exc:
        raise RuntimeError("Install huggingface_hub or configure a local input path.") from exc

    token = hf_token()
    try:
        return Path(hf_hub_download(
            repo_id=repo, filename=filename, repo_type=repo_type, revision=revision, token=token
        ))
    except GatedRepoError as exc:
        raise RuntimeError(
            f"{repo} is gated. Enable the Kaggle secret HF or HF_TOKEN with a read token "
            "from a Hugging Face account that has access to the repo, then rerun. "
            "Alternatively, attach the files and configure their local input path."
        ) from exc
    except Exception as exc:
        raise RuntimeError(
            f"Could not fetch {repo}/{filename}. Check the Kaggle Internet setting, "
            "Hub access and the file path, or configure a local input path above."
        ) from exc

def data_path(folder, filename):
    relative = Path(folder) / filename
    if LOCAL_DATA_ROOT is not None:
        return Path(LOCAL_DATA_ROOT) / relative
    for root in (Path("Phase_7_PCR"), Path.cwd()):
        if (root / relative).is_file():
            return root / relative
    return hub_file(DATA_REPO, f"Phase_7_PCR/{relative.as_posix()}", "dataset", DATA_REVISION)

def unique_pairs(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"Duplicate JSON key: {key!r}")
        result[key] = value
    return result

def reject_constant(value):
    raise ValueError(f"Invalid JSON constant: {value}")

def load_mapping(path):
    with path.open(encoding="utf-8") as stream:
        value = json.load(stream, object_pairs_hook=unique_pairs, parse_constant=reject_constant)
    if not isinstance(value, dict) or not value or any(not key.strip() for key in value):
        raise ValueError(f"{path}: expected a nonempty object keyed by case ID")
    return value

def sentences(record, case_id):
    text = record.get("text") if isinstance(record, dict) else record
    if isinstance(text, str):
        return [text]
    if isinstance(text, list) and all(isinstance(line, str) for line in text):
        return text
    raise ValueError(f"Case {case_id}: expected text as a string or list of sentences")

def percentiles(values):
    ordered = sorted(values)
    return {f"p{p}": ordered[round((len(ordered) - 1) * p / 100)] for p in (0, 50, 90, 95, 99, 100)}

WORD = re.compile(r"[^\W_]+")
'''
(RUN_ROOT / 'data_helpers.py').write_text(source, encoding='utf-8')
print('Prepared data_helpers.py')

In [ ]:
source = r'''"""Prepare split-local document tensors, BM25 scores and training groups."""
import argparse
from hashlib import sha256
from pathlib import Path
import time

import numpy as np
from collections import Counter
import torch

import data_helpers as source
import pcr_lexical as lexical
from bpe_tokenizer import PureByteLevelBPETokenizer
from pcr_worker import atomic_json, metrics, prediction, rank_scores


def document_chunks(text, tokenizer, length):
    tokens = tokenizer.encode(text)
    width = length - 2
    last = max(0, len(tokens) - width)
    offsets = list(range(0, last + 1, WINDOW_STRIDE))
    if offsets[-1] != last:
        offsets.append(last)
    if len(offsets) > MAX_WINDOWS:
        indices = np.linspace(0, len(offsets) - 1, MAX_WINDOWS).round().astype(int)
        offsets = [offsets[index] for index in indices]
    result = np.full((MAX_WINDOWS, length), tokenizer.pad_token_id, dtype=np.int32)
    for row, offset in enumerate(offsets):
        window = [tokenizer.bos_token_id, *tokens[offset:offset + width], tokenizer.eos_token_id]
        result[row, :len(window)] = window
    return result

def prepare_split(folder, stem, tokenizer, length, cached_chunks):
    queries = source.load_mapping(source.data_path(folder, f"{stem}.json"))
    candidates = source.load_mapping(source.data_path(folder, f"{stem}_candidates.json"))
    if SMOKE_MODE:
        queries, candidates = lexical.smoke_subset(queries, candidates, folder)
    qkeys, ckeys = sorted(queries), sorted(candidates)
    ccols = {key: i for i, key in enumerate(ckeys)}
    documents = []
    documents_by_id = {}

    def get_index(key, text):
        # Shared IDs with differing text keep their separate split-local representations.
        versions = documents_by_id.setdefault(key, [])
        for prior_text, index in versions:
            if prior_text == text:
                return index
        index = len(documents)
        documents.append(text)
        versions.append((text, index))
        return index

    qtext = ["\n".join(source.sentences(queries[key], key)) for key in qkeys]
    ctext = ["\n".join(source.sentences(candidates[key], key)) for key in ckeys]
    q_doc = np.array([get_index(key, text) for key, text in zip(qkeys, qtext)])
    c_doc = np.array([get_index(key, text) for key, text in zip(ckeys, ctext)])
    signals = lexical.features(qtext, ctext, SMOKE_MODE)
    scores = signals["doc_bm25"]
    same_case = np.equal.outer(qkeys, ckeys)
    if (len(ckeys) - same_case.sum(1) < 7).any():
        raise ValueError(f"{folder} lacks seven non-self candidates")
    gold = np.zeros_like(same_case)
    for row, key in enumerate(qkeys):
        record = queries[key]
        if folder == "Test":
            if "relevant_candidates" in record:
                raise ValueError("Unexpected Test labels")
            continue
        labels = record["relevant_candidates"]
        if not isinstance(labels, list):
            raise ValueError(f"Invalid gold labels for {key}")
        for label in labels:
            if isinstance(label, str) and label.strip():
                if label not in ccols or label == key or gold[row, ccols[label]]:
                    raise ValueError(f"Invalid, self, or duplicate gold link for {key}")
                gold[row, ccols[label]] = True

    hard = []
    if folder != "Test":
        for i in range(len(qkeys)):
            available = np.flatnonzero(~(same_case[i] | gold[i]))
            doc_order = available[np.argsort(-scores[i, available], kind="stable")[:12]]
            passage_order = available[np.argsort(-signals["passage_bm25"][i, available],
                                                 kind="stable")[:12]]
            hard.append(np.union1d(doc_order, passage_order))

    chunks = []
    for text in documents:
        # Pure tokenization cache keyed on the full text digest across splits.
        digest = sha256(text.encode("utf-8")).digest()
        if digest not in cached_chunks:
            cached_chunks[digest] = document_chunks(text, tokenizer, length)
        chunks.append(cached_chunks[digest])
    result = dict(ids=torch.from_numpy(np.stack(chunks)), q_doc=q_doc, c_doc=c_doc,
                  query_keys=qkeys, candidate_keys=ckeys, gold=gold, bm25=scores, hard=hard,
                  self=same_case, lexical=signals,
                  query_chars=np.array(list(map(len, qtext))),
                  candidate_chars=np.array(list(map(len, ctext))))
    if folder == "Train":
        eligible = np.flatnonzero(gold.any(1))
        if len(eligible) < 2:
            raise ValueError("Need at least two Train queries with positives")
        order = np.random.default_rng(781).permutation(eligible)
        holdout = max(1, round(.35 * len(order)))
        result["ranker_rows"] = np.sort(order[:holdout])
        result["neural_rows"] = np.sort(order[holdout:])
    return result


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", required=True, type=Path)
    parser.add_argument("--data", type=Path)
    parser.add_argument("--tokenizer", type=Path)
    parser.add_argument("--length", type=int, default=768)
    parser.add_argument("--stride", type=int, default=640)
    parser.add_argument("--windows", type=int, default=12)
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    if args.length < 16 or args.length > 1280:
        raise ValueError("Chunk size must be between 16 and 1280")
    if not 1 <= args.stride <= args.length - 2 or args.windows < 2:
        raise ValueError("Stride must cover the text and window cap must be at least two")
    global WINDOW_STRIDE, MAX_WINDOWS, SMOKE_MODE
    WINDOW_STRIDE, MAX_WINDOWS, SMOKE_MODE = args.stride, args.windows, args.smoke
    source.LOCAL_DATA_ROOT = args.data
    directory = args.tokenizer
    if directory is None:
        vocab = source.hub_file(source.TOKENIZER_REPO, "vocab.json", "model")
        merges = source.hub_file(source.TOKENIZER_REPO, "merges.json", "model")
        if vocab.parent != merges.parent:
            raise RuntimeError("Tokenizer files came from different revisions")
        directory = vocab.parent
    tokenizer = PureByteLevelBPETokenizer.load(str(directory))
    if len(tokenizer.encoder) != 33000 or max(tokenizer.encoder.values()) != 32999:
        raise ValueError("Expected the original 33,000-entry tokenizer")

    bundle = {"pad": tokenizer.pad_token_id, "bos": tokenizer.bos_token_id,
              "eos": tokenizer.eos_token_id}
    cache = {}
    for folder, stem in source.SPLITS.items():
        started = time.monotonic()
        split = prepare_split(folder, stem, tokenizer, args.length, cache)
        bundle[folder] = split
        print(f"{folder} · {len(split['query_keys'])} queries · {len(split['candidate_keys'])} candidates · "
              f"{len(split['ids'])} documents · {int(split['ids'][..., 0].ne(tokenizer.pad_token_id).sum()):,} windows · {time.monotonic() - started:.1f}s", flush=True)
    dev = bundle["Dev"]
    ranks = rank_scores(dev["bm25"], dev, 0.0)
    atomic_json(args.root / "bm25_dev.json", metrics(ranks, dev["gold"]))
    test = bundle["Test"]
    ranks = rank_scores(test["bm25"], test, 0.0)
    atomic_json(args.root / "bm25_prediction.json", prediction(ranks, test))
    temporary = args.root / "inputs.tmp"
    torch.save(bundle, temporary)
    temporary.replace(args.root / "inputs.pt")


if __name__ == "__main__":
    main()
'''
(RUN_ROOT / 'pcr_prepare.py').write_text(source, encoding='utf-8')
print('Prepared pcr_prepare.py')

In [ ]:
source = r'''"""Phase 7 retrieval training and export. Invoked by the Kaggle notebook."""
import argparse
import math
from contextlib import nullcontext
import json
from pathlib import Path
import time

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as activation_checkpoint

from llama_scratch import Config, LlamaLM


class LowRank(nn.Module):
    def __init__(self, linear, multiplicative, rank=8):
        super().__init__()
        self.linear = linear
        self.multiplicative = multiplicative
        width = linear.out_features if multiplicative else linear.in_features
        self.down = nn.Linear(width, rank, bias=False)
        self.up = nn.Linear(rank, linear.out_features, bias=False)
        nn.init.zeros_(self.up.weight)

    def forward(self, x):
        y = self.linear(x)
        update = self.up(self.down(y if self.multiplicative else x))
        return y + update


class Retriever(nn.Module):
    def __init__(self, checkpoint, method, pad):
        super().__init__()
        cfg = Config(**checkpoint["cfg"])
        cfg.attn_impl = "sdpa"
        self.base = LlamaLM(cfg)
        self.base.load_state_dict(checkpoint["model"], strict=True)
        self.base.requires_grad_(False)
        self.pad = pad
        if method != "lorma":
            raise ValueError(f"Unexpected retrieval method: {method}")
        for block in self.base.layers:
            block.attn.wq = LowRank(block.attn.wq, multiplicative=True)
            block.attn.wv = LowRank(block.attn.wv, multiplicative=True)

        self.head = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        nn.init.eye_(self.head.weight)

    def hidden_states(self, ids):
        length = ids.size(1)
        if length > self.base.cfg.max_seq_len:
            raise ValueError("Input exceeds the checkpoint's context length")
        cos = self.base.rope_cos[:length]
        sin = self.base.rope_sin[:length]
        hidden = self.base.tok_emb(ids)
        for block in self.base.layers:
            if self.training:
                hidden, _ = activation_checkpoint(
                    block, hidden, cos, sin, use_reentrant=False, preserve_rng_state=False,
                )
            else:
                hidden, _ = block(hidden, cos, sin)
        return self.base.norm(hidden)

    def window_embeddings(self, ids):
        valid = ids.ne(self.pad)
        hidden = self.hidden_states(ids)
        pooled = (hidden.float() * valid[..., None]).sum(1)
        pooled = pooled / valid.sum(1).clamp(min=1)[:, None]
        return F.normalize(self.head(pooled).float(), dim=-1)


def atomic_json(path, data):
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(data, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    tmp.replace(path)


def atomic_weights(path, model):
    tmp = path.with_suffix(".tmp")
    weights = {name: p.detach().cpu() for name, p in model.named_parameters() if p.requires_grad}
    torch.save(weights, tmp)
    tmp.replace(path)


def cast_frozen(model, device):
    if device.type == "cuda":
        for parameter in model.parameters():
            if not parameter.requires_grad:
                parameter.data = parameter.data.half()


def build_model(checkpoint, method, pad, device):
    model = Retriever(checkpoint, method, pad)
    cast_frozen(model, device)
    return model.to(device)


def load_report(output):
    return json.loads((output / "dev.json").read_text(encoding="utf-8"))


def restore(model, path):
    weights = torch.load(path, map_location="cpu", weights_only=True)
    trainable = {name for name, p in model.named_parameters() if p.requires_grad}
    if set(weights) != trainable:
        raise ValueError(f"Unexpected checkpoint keys in {path}")
    model.load_state_dict(weights, strict=False)


def amp(device):
    return torch.autocast("cuda", dtype=torch.float16) if device.type == "cuda" else nullcontext()


def normalize_rows(scores, excluded):
    finite = np.where(excluded, np.nan, scores)
    lo = np.nanmin(finite, axis=1, keepdims=True)
    hi = np.nanmax(finite, axis=1, keepdims=True)
    return (np.where(excluded, lo, scores) - lo) / np.maximum(hi - lo, 1e-8)


def rank_scores(dense, split, weight):
    excluded = split["self"]

    def reciprocal(scores):
        order = np.argsort(-np.where(excluded, -np.inf, scores), axis=1, kind="stable")
        return 1.0 / (41 + np.argsort(order, axis=1, kind="stable"))

    if weight == 0:
        mixed = split["bm25"].copy()
    elif weight == 1:
        mixed = dense.copy()
    else:
        lexical = reciprocal(split["bm25"]) + reciprocal(split["lexical"]["passage_bm25"])
        mixed = weight * reciprocal(dense) + (1 - weight) * .5 * lexical
    mixed[excluded] = -np.inf
    return np.argsort(-mixed, axis=1, kind="stable")[:, :7]


def metrics(ranks, gold):
    tp = int(np.take_along_axis(gold, ranks, axis=1).sum())
    predicted = ranks.size
    relevant = int(gold.sum())
    return dict(f1=2 * tp / (predicted + relevant), precision=tp / predicted,
                recall=tp / max(1, relevant), tp=tp)


def sample_windows(ids, pad, rng, count=2):
    valid = ids[..., 0].ne(pad).numpy()
    indices = np.zeros((len(ids), count), dtype=np.int64)
    mask = np.zeros_like(indices, dtype=bool)
    for row, available in enumerate(valid):
        choices = np.flatnonzero(available)
        picks = rng.choice(choices, min(count, len(choices)), replace=False)
        indices[row, :len(picks)] = picks
        mask[row, :len(picks)] = True
    return indices, mask


def group_loss(scores, gold, excluded):
    negatives = ~(gold | excluded)
    pairs = gold[:, :, None] & negatives[:, None, :]
    # Each known positive is compared directly with the negatives for its query.
    deltas = (scores[:, None, :] - scores[:, :, None] + .05) / .10
    loss = F.softplus(deltas).masked_fill(~pairs, 0)
    counts = pairs.sum(dim=(1, 2))
    return ((loss.sum(dim=(1, 2)) / counts.clamp_min(1)).sum()
            / counts.gt(0).sum().clamp_min(1))


def passage_scores(q, c, q_valid, c_valid, statistics=False):
    cosine = torch.einsum("qxh,cyh->qcxy", q, c)
    valid = q_valid[:, None, :, None] & c_valid[None, :, None, :]
    cosine = cosine.masked_fill(~valid, -2.0)
    flat = cosine.flatten(2)
    first, index = flat.max(2)
    q_position = torch.arange(q.size(1), device=q.device)
    c_position = torch.arange(c.size(1), device=c.device)
    different_q = q_position[None, None, :, None] != (index // c.size(1))[:, :, None, None]
    different_c = c_position[None, None, None, :] != (index % c.size(1))[:, :, None, None]
    second = cosine.masked_fill(~(different_q & different_c), -2.0).flatten(2).amax(2)
    score = torch.where(second > -1.5, (first + second) / 2, first)
    if statistics:
        q_best = cosine.amax(-1).masked_fill(~q_valid[:, None, :], 0)
        coverage = q_best.sum(-1) / q_valid.sum(-1).clamp(min=1)[:, None]
        return torch.stack((score, first, coverage), dim=-1)
    return score


def encode(model, ids, device, batch, deadline):
    valid = ids[..., 0].ne(model.pad)
    positions = valid.nonzero()
    result = torch.zeros((*valid.shape, model.base.cfg.d_model), device=device, dtype=torch.float16)
    model.eval()
    with torch.inference_mode(), amp(device):
        for offset in range(0, len(positions), batch):
            if time.monotonic() >= deadline:
                raise TimeoutError("Window encoding reached the stage deadline")
            doc, window = positions[offset:offset + batch].unbind(1)
            embeddings = model.window_embeddings(ids[doc, window].to(device))
            result[doc.to(device), window.to(device)] = embeddings.half()
    return result


def all_scores(vectors, valid, split, deadline):
    device = vectors.device
    queries = torch.as_tensor(split["q_doc"], device=device)
    candidates = torch.as_tensor(split["c_doc"], device=device)
    candidate_vectors = vectors[candidates]
    candidate_valid = valid[candidates]
    results = []
    for query in queries.split(1):
        if time.monotonic() >= deadline:
            raise TimeoutError("Passage scoring reached the stage deadline")
        results.append(passage_scores(vectors[query], candidate_vectors, valid[query], candidate_valid))
    return torch.cat(results).float().cpu().numpy()


def fixed_dev_loss(vectors, split, model, args):
    rng = np.random.default_rng(args.seed + 1)
    device = vectors.device
    indices, masks = sample_windows(split["ids"], model.pad, rng)
    rows = torch.arange(len(indices), device=device)[:, None]
    subset = vectors[rows, torch.from_numpy(indices).to(device)].float()
    mask = torch.from_numpy(masks).to(device)
    eligible = np.flatnonzero(split["gold"].any(1))
    losses = []
    for offset in range(0, len(eligible), args.query_batch):
        rows = eligible[offset:offset + args.query_batch]
        cols = choose_batch(split, rows, rng)
        q, c = split["q_doc"][rows], split["c_doc"][cols]
        scores = passage_scores(subset[q], subset[c], mask[q], mask[c])
        targets = torch.from_numpy(split["gold"][np.ix_(rows, cols)]).to(device)
        excluded = torch.from_numpy(split["self"][np.ix_(rows, cols)]).to(device)
        losses.append((float(group_loss(scores, targets, excluded)), len(rows)))
    return sum(loss * size for loss, size in losses) / sum(size for _, size in losses)


def validate(model, split, device, args):
    started = time.monotonic()
    with torch.inference_mode():
        vectors = encode(model, split["ids"], device, args.encode_batch, args.deadline)
        valid = split["ids"][..., 0].ne(model.pad).to(device)
        dense = all_scores(vectors, valid, split, args.deadline)
        loss = fixed_dev_loss(vectors, split, model, args)
    gold = split["gold"]
    weight = 1 / 3
    result = metrics(rank_scores(dense, split, weight), gold)
    dense_result = metrics(rank_scores(dense, split, 1.), gold)
    return dict(**result, dense_f1=dense_result["f1"], dense_weight=weight,
                val_loss=loss, dev_seconds=time.monotonic() - started)


def update(model, split, rows, rng, device, optimizer, scaler):
    cols = choose_batch(split, rows, rng)
    docs = np.concatenate((split["q_doc"][rows], split["c_doc"][cols]))
    ids = split["ids"][docs]
    windows, valid = sample_windows(ids, model.pad, rng)
    ids = ids[torch.arange(len(ids))[:, None], torch.from_numpy(windows)].to(device)
    valid = torch.from_numpy(valid).to(device)
    mask = torch.from_numpy(split["self"][np.ix_(rows, cols)]).to(device)
    gold = torch.from_numpy(split["gold"][np.ix_(rows, cols)]).to(device)
    optimizer.zero_grad(set_to_none=True)
    with amp(device):
        embeddings = model.window_embeddings(ids.flatten(0, 1))
    embeddings = embeddings.reshape(*valid.shape, -1)
    count = len(rows)
    scores = passage_scores(embeddings[:count], embeddings[count:], valid[:count], valid[count:])
    loss = group_loss(scores, gold, mask)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
    scaler.step(optimizer)
    scaler.update()
    return float(loss.detach())

def choose_batch(split, rows, rng):
    candidates = []
    for row in rows:
        positives = np.flatnonzero(split["gold"][row])
        selected = rng.choice(positives, size=min(2, len(positives)), replace=False)
        lexical = split["hard"][row]
        if "mined" in split:
            dense = split["mined"][row]
            dense = rng.choice(dense, size=min(2, len(dense)), replace=False)
            lexical = lexical[~np.isin(lexical, dense)]
            hard = np.concatenate((dense, rng.choice(lexical, size=min(1, len(lexical)),
                                                      replace=False)))
        else:
            hard = rng.choice(lexical, size=min(3, len(lexical)), replace=False)
        candidates.extend(selected)
        candidates.extend(hard)
    return np.unique(candidates)


def mine(model, split, device, args, deadline):
    started = time.monotonic()
    rows = split["neural_rows"]
    view = {**split, "q_doc": split["q_doc"][rows]}
    with torch.inference_mode():
        vectors = encode(model, split["ids"], device, args.encode_batch, deadline)
        valid = split["ids"][..., 0].ne(model.pad).to(device)
        dense = all_scores(vectors, valid, view, deadline)
    output = {}
    mined = [np.empty(0, dtype=np.int64) for _ in split["query_keys"]]
    for row, i in enumerate(rows):
        pool = np.flatnonzero(~(split["gold"][i] | split["self"][i]))
        order = pool[np.argsort(-dense[row, pool], kind="stable")[:20]]
        mined[i] = order
        output[split["query_keys"][i]] = [split["candidate_keys"][j] for j in order]
    split["mined"] = mined
    atomic_json(args.root / "hard_negatives.json", output)
    print(f"Mined up to 20 dense negatives per retriever Train query in "
          f"{time.monotonic()-started:.1f}s", flush=True)


class LossPatience:
    def __init__(self, patience=3, min_delta=.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best = math.inf
        self.stale = 0

    def observe(self, loss):
        if not math.isfinite(loss):
            raise ValueError(f"Non-finite Dev pairwise loss: {loss}")
        if loss < self.best - self.min_delta:
            self.best = loss
            self.stale = 0
        else:
            self.stale += 1
        return self.stale >= self.patience


def train(args, bundle, checkpoint, device):
    output = args.root / "lorma"
    output.mkdir(exist_ok=True)
    started = time.monotonic()
    rng = np.random.default_rng(args.seed)
    model = build_model(checkpoint, "lorma", bundle["pad"], device)
    train_data, dev_data = bundle["Train"], bundle["Dev"]
    eligible = train_data["neural_rows"]
    if not len(eligible):
        raise ValueError("No eligible Train queries for LoRMA+")
    initial = validate(model, dev_data, device, args)
    atomic_json(args.root / "initial_dev.json", initial)
    print(f"Initial Dev loss {initial['val_loss']:.4f} · F1@7 {initial['f1']:.4f} · "
          f"training {len(eligible)} queries · ranker holdout {len(train_data['ranker_rows'])}",
          flush=True)
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=1.5e-4, fused=device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    early = LossPatience()
    eval_reserve = max(3, initial["dev_seconds"] * 1.4 + 4)
    schedule_start = time.monotonic()
    next_check = schedule_start + args.validation_seconds
    history, losses = [], []
    step = seen = 0
    average_step = 0.
    best_key = best_file = None
    reason = "deadline"
    order = rng.permutation(eligible)
    offset = 0
    mined = False

    while time.monotonic() + eval_reserve + max(1., 1.5 * average_step) < args.deadline:
        if offset >= len(order):
            order = rng.permutation(eligible)
            offset = 0
        rows = order[offset:offset + args.query_batch]
        offset += len(rows)
        tick = time.monotonic()
        fraction = min(1.0, (tick - schedule_start) / max(1., args.deadline - schedule_start))
        decay = .2 + .8 * .5 * (1 + math.cos(math.pi * fraction))
        warmup = 5 if args.smoke else 80
        for group in optimizer.param_groups:
            group["lr"] = 1.5e-4 * min(1., (step + 1) / warmup) * decay
        model.train()
        losses.append(update(model, train_data, rows, rng, device, optimizer, scaler))
        step += 1
        seen += len(rows)
        elapsed = time.monotonic() - tick
        average_step = .9 * average_step + .1 * elapsed if average_step else elapsed

        now = time.monotonic()
        first_epoch = seen == len(eligible)
        smoke_done = args.smoke and seen >= 2 * len(eligible)
        final_check = now + eval_reserve + max(1., 1.5 * average_step) >= args.deadline
        if not (first_epoch or smoke_done or final_check or now >= next_check):
            continue
        if now + eval_reserve > args.deadline and best_file is not None:
            break
        result = validate(model, dev_data, device, args)
        stop = early.observe(result["val_loss"])
        result.update(method="lorma", step=step, epochs=seen / len(eligible),
                      train_loss=float(np.mean(losses)), loss_stale_checks=early.stale)
        history.append(result.copy())
        key = (result["f1"], -result["val_loss"])
        if best_key is None or key > best_key:
            new_file = output / f"checkpoint_{step}.pt"
            result["weights_file"] = new_file.name
            atomic_weights(new_file, model)
            atomic_json(output / "dev.json", result)
            if best_file is not None:
                best_file.unlink(missing_ok=True)
            best_key, best_file = key, new_file
        atomic_json(output / "history.json", history)
        now = time.monotonic()
        print(f"LoRMA+ step {step} · epoch {result['epochs']:.2f} · elapsed {now-started:.0f}s · "
              f"time left {max(0,args.deadline-now):.0f}s · train loss {result['train_loss']:.4f} · "
              f"Dev loss {result['val_loss']:.4f} · F1@7 {result['f1']:.4f} · "
              f"stale {early.stale}/{early.patience}", flush=True)
        losses.clear()
        eval_reserve = max(eval_reserve, result["dev_seconds"] * 1.4 + 4)
        next_check = now + args.validation_seconds
        if stop:
            reason = "three_dev_checks_without_loss_improvement"
            break
        if smoke_done:
            reason = "smoke_complete"
            break
        if final_check:
            break
        if first_epoch and not mined and now + max(20, 4 * eval_reserve) < args.deadline:
            try:
                mine(model, train_data, device, args, args.deadline - eval_reserve)
                mined = True
            except TimeoutError:
                break

    atomic_json(output / "training.json", dict(
        epochs_trained=seen / len(eligible), steps=step, stop_reason=reason,
        mining_completed=mined, dev_checks=len(history),
        best_dev_loss=None if not history else min(row["val_loss"] for row in history),
    ))
    print(f"Retriever finished · {reason} · {seen / len(eligible):.2f} epochs", flush=True)

def prediction(ranks, split):
    result = {
        query: [split["candidate_keys"][int(i)] for i in row]
        for query, row in zip(split["query_keys"], ranks, strict=True)
    }
    pool = set(split["candidate_keys"])
    if set(result) != set(split["query_keys"]):
        raise ValueError("Prediction query IDs do not match the test split")
    if not all(len(ids) == len(set(ids)) == 7 and set(ids) <= pool for ids in result.values()):
        raise ValueError("Invalid ranked candidate IDs")
    return result


def export(args, bundle, checkpoint, device):
    for method in args.method.split(","):
        output = args.root / method
        report = load_report(output)
        model = build_model(checkpoint, method, bundle["pad"], device)
        restore(model, output / report["weights_file"])
        split = bundle["Test"]
        started = time.monotonic()
        with torch.inference_mode():
            vectors = encode(model, split["ids"], device, args.encode_batch, args.deadline)
            valid = split["ids"][..., 0].ne(model.pad).to(device)
            dense = all_scores(vectors, valid, split, args.deadline)
        ranks = rank_scores(dense, split, report["dense_weight"])
        atomic_json(output / "prediction.json", prediction(ranks, split))
        print(f"{method} · test export {time.monotonic() - started:.1f}s", flush=True)
        del model


def load_inputs(args):
    bundle = torch.load(args.root / "inputs.pt", weights_only=False, map_location="cpu")
    checkpoint = torch.load(args.checkpoint, weights_only=True, map_location="cpu")
    config = checkpoint["cfg"]
    if (config.get("vocab_size") != 33000 or config.get("attn_type", "gqa") != "gqa"
            or config.get("pos_enc", "rope") != "rope"):
        raise ValueError(f"Expected the original Phase 2 v4 checkpoint config: {config}")
    if bundle["Train"]["ids"].size(-1) > config["max_seq_len"]:
        raise ValueError("Configured passage length exceeds the Phase 2 v4 context")
    print(f"Phase 2 v4 · GQA/RoPE · {config['n_layers']} layers · width {config['d_model']} · "
          f"context {config['max_seq_len']} · parallel attention {config.get('parallel_attn', True)}",
          flush=True)
    return bundle, checkpoint


def main():
    p = argparse.ArgumentParser()
    p.add_argument("action", choices=("train", "export"))
    p.add_argument("method")
    p.add_argument("--root", type=Path, required=True)
    p.add_argument("--checkpoint", type=Path, required=True)
    p.add_argument("--deadline", type=float, required=True)
    p.add_argument("--encode-batch", type=int, default=16)
    p.add_argument("--query-batch", type=int, default=3)
    p.add_argument("--seed", type=int, default=781)
    p.add_argument("--validation-seconds", type=float, default=180)
    p.add_argument("--smoke", action="store_true")
    args = p.parse_args()
    torch.set_num_threads(1)
    torch.manual_seed(args.seed)
    device = torch.device("cuda:0")
    if torch.cuda.device_count() != 1:
        raise RuntimeError("Expose exactly one CUDA GPU to the v4 worker")
    bundle, checkpoint = load_inputs(args)
    if args.action == "train":
        train(args, bundle, checkpoint, device)
    else:
        export(args, bundle, checkpoint, device)


if __name__ == "__main__":
    main()
'''
(RUN_ROOT / 'pcr_worker.py').write_text(source, encoding='utf-8')
print('Prepared pcr_worker.py')

In [ ]:
source = r'''"""Split-local document and passage BM25 features."""
from collections import Counter
import re
import time

import numpy as np
from scipy.sparse import csr_matrix

WORD = re.compile(r"[^\W_]+")
PLACEHOLDERS = re.compile(r"<[A-Z_]+>")
SECTION = re.compile(r"\b(section|sec|article|art|rule|order)\.?\s+(\d+[a-z]?(?:\s*\([0-9a-z]+\))*)", re.I)
ACT = re.compile(r"\b(?:[a-z][a-z-]*\s+){1,5}(?:act|code|constitution)\b(?:\s*,?\s*\d{4})?", re.I)
KINDS = {"sec": "section", "art": "article"}
STOP = set("a an and are as at be been by for from has have he her in is it its of on or that the their this to was were which with would".split())


def words(text):
    return [word for word in WORD.findall(PLACEHOLDERS.sub(" ", text).casefold()) if word not in STOP]


def sample(sequence, limit):
    if len(sequence) <= limit:
        return sequence
    return [sequence[i] for i in np.linspace(0, len(sequence)-1, limit).round().astype(int)]


def passages(tokens, limit):
    # Reserve the last passage and spread the rest across the judgment.
    starts = list(range(0, max(1, len(tokens)-255), 192))
    tail = max(0, len(tokens)-256)
    if starts[-1] != tail:
        starts.append(tail)
    return [tokens[start:start+256] for start in sample(starts, limit)]


class BM25:
    def __init__(self, documents):
        self.vocabulary = {}
        self.count = len(documents)
        values, cols, offsets = [], [], [0]
        for document in documents:
            for term, count in Counter(document).items():
                cols.append(self.vocabulary.setdefault(term, len(self.vocabulary)))
                values.append(count)
            offsets.append(len(cols))
        counts = csr_matrix((np.asarray(values, dtype=np.float32), cols, offsets),
                            shape=(self.count, len(self.vocabulary)))
        size = np.asarray(counts.sum(axis=1)).ravel()
        df = np.asarray(counts.getnnz(axis=0))
        self.idf = np.log1p((self.count - df + .5) / (df + .5)).astype(np.float32)
        norm = 1.2 * (.25 + .75 * size / max(float(size.mean()), 1.))
        denominator = counts.data + np.repeat(norm, np.diff(counts.indptr))
        weights = 2.2 * counts.data * self.idf[counts.indices] / denominator
        self.transposed = csr_matrix((weights, counts.indices, counts.indptr),
                                     shape=counts.shape).T.tocsr()

    def query(self, tokens, max_terms=192):
        counts = Counter(tokens)
        eligible = [(self.vocabulary[word], count) for word, count in counts.items()
                    if word in self.vocabulary]
        eligible.sort(key=lambda pair: (-float(self.idf[pair[0]]) * min(pair[1], 3), pair[0]))
        eligible = eligible[:max_terms]
        cols = [col for col, _ in eligible]
        values = np.asarray([1 + .25 * (min(count, 4) - 1) for _, count in eligible],
                            dtype=np.float32)
        q = csr_matrix((values, cols, [0, len(cols)]), shape=(1, len(self.vocabulary)))
        return (q @ self.transposed).tocsr()


def legal_references(text):
    sections = {f"{KINDS.get(kind.lower(), kind.lower())}:{re.sub(chr(32), '', number.lower())}"
                for kind, number in SECTION.findall(text)}
    acts = set()
    for match in ACT.findall(text.casefold()):
        parts = re.sub(r"[^a-z0-9 ]", " ", match).split()
        if parts:
            acts.add(" ".join(parts[-5:]))
    return sections, acts


def overlap(queries, candidates):
    postings = {}
    for i, identifiers in enumerate(candidates):
        for term in identifiers:
            postings.setdefault(term, []).append(i)
    result = np.zeros((len(queries), len(candidates)), dtype=np.float32)
    for i, identifiers in enumerate(queries):
        for term in identifiers:
            cols = postings.get(term)
            if cols:
                result[i, cols] += np.log1p(len(candidates) / len(cols))
    return result


def features(queries, candidates, smoke):
    started = time.monotonic()
    q_tokens = [words(text) for text in queries]
    c_tokens = [words(text) for text in candidates]
    docs = BM25([sample(tokens, 8192) for tokens in c_tokens])
    doc_scores = np.vstack([docs.query(tokens).toarray()[0] for tokens in q_tokens])
    del docs

    snippets = []
    owners = []
    for i, tokens in enumerate(c_tokens):
        views = passages(tokens, 6 if smoke else 20)
        snippets.extend(views)
        owners.extend([i] * len(views))
    owners = np.asarray(owners)
    index = BM25(snippets)
    del snippets, c_tokens
    passage_scores = np.zeros_like(doc_scores)
    for i, tokens in enumerate(q_tokens):
        # Compare the whole-case vocabulary and four sections independently.
        for query in [tokens, *passages(tokens, 4)]:
            scores = index.query(query, max_terms=96)
            np.maximum.at(passage_scores[i], owners[scores.indices], scores.data)
    del index, q_tokens

    q_ref = [legal_references(text) for text in queries]
    c_ref = [legal_references(text) for text in candidates]
    q_sec, q_act = zip(*q_ref, strict=True)
    c_sec, c_act = zip(*c_ref, strict=True)
    print(f"Lexical features · {len(owners):,} candidate passages · "
          f"{time.monotonic()-started:.1f}s", flush=True)
    return {
        "doc_bm25": doc_scores,
        "passage_bm25": passage_scores,
        "section_overlap": overlap(q_sec, c_sec),
        "act_overlap": overlap(q_act, c_act),
    }


def smoke_subset(queries, candidates, folder):
    rng = np.random.default_rng({"Train": 781, "Dev": 782, "Test": 783}[folder])
    keys = sorted(queries)
    if folder != "Test":
        eligible = [key for key in keys if any(
            isinstance(label, str) and label in candidates and label != key
            for label in queries[key]["relevant_candidates"]
        )]
        keys = eligible or keys
    q_limit = 48 if folder == "Train" else 16
    picked = rng.choice(keys, min(q_limit, len(keys)), replace=False).tolist()
    keep = {key for key in picked if key in candidates}
    if folder != "Test":
        keep.update(label for key in picked for label in queries[key]["relevant_candidates"]
                    if isinstance(label, str) and label in candidates)
    others = sorted(set(candidates) - keep)
    add = max(0, min(len(others), 160 - len(keep)))
    keep.update(rng.choice(others, add, replace=False).tolist())
    return {key: queries[key] for key in picked}, {key: candidates[key] for key in sorted(keep)}
'''
(RUN_ROOT / 'pcr_lexical.py').write_text(source, encoding='utf-8')
print('Prepared pcr_lexical.py')

In [ ]:
source = r'''"""Phase 7 v4 learned fusion using grouped LambdaMART."""
import argparse
import importlib
from pathlib import Path
import subprocess
import sys
import time

import numpy as np
import torch

import pcr_worker as pcr


FEATURE_NAMES = [
    "doc_bm25", "passage_bm25", "dense_two_matches", "dense_best_match", "dense_query_coverage",
    "doc_reciprocal_rank", "passage_reciprocal_rank", "dense_reciprocal_rank",
    "section_overlap", "act_overlap", "candidate_length_log", "length_ratio_log",
    "hybrid_rrf",
]


def ranking(scores, excluded):
    order = np.argsort(-np.where(excluded, -np.inf, scores), axis=1, kind="stable")
    ranks = np.argsort(order, axis=1, kind="stable") + 1
    return order, ranks


def score_passages(vectors, valid, query_docs, candidate_docs, deadline):
    device = vectors.device
    c = vectors[torch.as_tensor(candidate_docs, device=device)]
    c_valid = valid[torch.as_tensor(candidate_docs, device=device)]
    results = []
    with torch.inference_mode():
        for query in torch.as_tensor(query_docs, device=device).split(1):
            if time.monotonic() >= deadline:
                raise TimeoutError("Dense passage scoring reached the deadline")
            stats = pcr.passage_scores(vectors[query], c, valid[query], c_valid, statistics=True)
            results.append(stats.squeeze(0).float())
        return torch.stack(results).cpu().numpy()


def build_features(split, query_rows, dense, depth):
    lexical = split["lexical"]
    doc = lexical["doc_bm25"][query_rows]
    passage = lexical["passage_bm25"][query_rows]
    excluded = split["self"][query_rows]
    gold = split["gold"][query_rows]
    dense2, dense1, coverage = np.moveaxis(dense, -1, 0)
    order_doc, rank_doc = ranking(doc, excluded)
    order_pass, rank_pass = ranking(passage, excluded)
    order_dense, rank_dense = ranking(dense2, excluded)
    r_doc, r_pass, r_dense = (1.0 / (40 + rank) for rank in (rank_doc, rank_pass, rank_dense))
    hybrid = r_doc + r_pass + r_dense
    hybrid_top = ranking(hybrid, excluded)[0][:, :7]
    cols_by_query = []
    features_by_query = []
    for row, source_row in enumerate(query_rows):
        count = min(depth, np.count_nonzero(~excluded[row]))
        shortlist = np.unique(np.concatenate((
            order_doc[row, :count], order_pass[row, :count], order_dense[row, :count], hybrid_top[row]
        )))
        q_size = max(1, split["query_chars"][source_row])
        c_size = np.maximum(1, split["candidate_chars"][shortlist])
        values = (
            doc[row, shortlist], passage[row, shortlist], dense2[row, shortlist],
            dense1[row, shortlist], coverage[row, shortlist], r_doc[row, shortlist],
            r_pass[row, shortlist], r_dense[row, shortlist],
            lexical["section_overlap"][source_row, shortlist],
            lexical["act_overlap"][source_row, shortlist],
            np.log1p(c_size), np.log(c_size / q_size), hybrid[row, shortlist],
        )
        cols_by_query.append(shortlist)
        features_by_query.append(np.column_stack(values).astype(np.float32))
    return dict(cols=cols_by_query, x=features_by_query, gold=gold, hybrid_top=hybrid_top)


def flatten(data, training):
    keep = []
    for row, cols in enumerate(data["cols"]):
        labels = data["gold"][row, cols]
        if not training or (labels.any() and not labels.all()):
            keep.append(row)
    if not keep:
        raise ValueError("No query groups with both positive and negative labels")
    return (np.concatenate([data["x"][row] for row in keep]),
            np.concatenate([data["gold"][row, data["cols"][row]] for row in keep]).astype(np.int32),
            [len(data["cols"][row]) for row in keep])


def rerank(pred, data, weight):
    if weight == 0:
        return data["hybrid_top"]
    ranks = []
    offset = 0
    for cols, features in zip(data["cols"], data["x"], strict=True):
        values = pred[offset:offset + len(cols)]
        model_rank = np.argsort(np.argsort(-values, kind="stable"), kind="stable") + 1
        base_rank = np.argsort(np.argsort(-features[:, -1], kind="stable"), kind="stable") + 1
        score = weight / (20 + model_rank) + (1 - weight) / (20 + base_rank)
        ranks.append(cols[np.argsort(-score, kind="stable")[:7]])
        offset += len(cols)
    if offset != len(pred):
        raise ValueError("Rank scores do not align with query groups")
    return np.stack(ranks)


def oracle(data):
    gold = data["gold"]
    retrieved = np.array([gold[row, cols].sum() for row, cols in enumerate(data["cols"])])
    total = int(gold.sum())
    denominator = 7 * len(gold) + total
    return dict(
        queries=len(gold), gold=total,
        mean_candidates=float(np.mean([len(cols) for cols in data["cols"]])),
        shortlist_recall=float(retrieved.sum() / max(1, total)),
        oracle_micro_f1_at_7=float(2 * np.minimum(7, retrieved).sum() / denominator),
    )


def get_lightgbm(deadline):
    try:
        return importlib.import_module("lightgbm")
    except ModuleNotFoundError as error:
        if error.name != "lightgbm":
            raise
        remaining = int(deadline - time.monotonic() - 30)
        if remaining < 15:
            raise TimeoutError("No time left to install LightGBM")
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "lightgbm"],
                       check=True, timeout=min(120, remaining))
        importlib.invalidate_caches()
        return importlib.import_module("lightgbm")


def fit_ranker(train, dev, output, args):
    lgb = get_lightgbm(args.deadline)
    train_x, train_y, train_groups = flatten(train, training=True)
    dev_x, dev_y, dev_groups = flatten(dev, training=False)
    print(f"LambdaMART · {len(train_groups)} held-out Train query groups · "
          f"{len(train_y):,} candidate pairs", flush=True)
    a = lgb.Dataset(train_x, label=train_y, group=train_groups, feature_name=FEATURE_NAMES)
    b = lgb.Dataset(dev_x, label=dev_y, group=dev_groups, reference=a)

    def dev_metric(pred, dataset):
        if time.monotonic() + 6 >= args.deadline:
            raise TimeoutError("LambdaMART reached the export reserve")
        f1 = pcr.metrics(rerank(pred, dev, 1.), dev["gold"])["f1"]
        return "micro_f1_at_7", f1, True

    model = lgb.train(
        {"objective": "lambdarank", "metric": "None", "lambdarank_truncation_level": 10,
         "learning_rate": .045, "num_leaves": 15, "min_data_in_leaf": 6 if args.smoke else 35,
         "lambda_l2": 3., "feature_fraction": .85, "seed": args.seed, "num_threads": 4,
         "verbosity": -1, "force_col_wise": True},
        a, num_boost_round=24 if args.smoke else 220, valid_sets=[b],
        feval=dev_metric, callbacks=[lgb.early_stopping(8 if args.smoke else 35, verbose=False)],
    )
    model.save_model(str(output / "lambda_model.txt"))
    preds = model.predict(dev_x, num_threads=4)
    ranked = [(pcr.metrics(rerank(preds, dev, weight), dev["gold"]), weight)
              for weight in (.35, .65, 1.0)]
    scores, weight = max(ranked, key=lambda item: item[0]["f1"])
    return model, dict(**scores, model_weight=weight, trees=model.best_iteration,
                       train_query_groups=len(train_groups))


def run(args, bundle, checkpoint, device):
    retrieval = pcr.load_report(args.root / "lorma")
    model = pcr.build_model(checkpoint, "lorma", bundle["pad"], device)
    pcr.restore(model, args.root / "lorma" / retrieval["weights_file"])
    splits = {}
    hybrid_dir = args.root / "hybrid"
    hybrid_dir.mkdir(exist_ok=True)
    for name in ("Test", "Dev", "Train"):
        started = time.monotonic()
        split = bundle[name]
        rows = split["ranker_rows"] if name == "Train" else np.arange(len(split["query_keys"]))
        vectors = pcr.encode(model, split["ids"], device, args.encode_batch, args.deadline)
        valid = split["ids"][..., 0].ne(bundle["pad"]).to(device)
        dense = score_passages(vectors, valid, split["q_doc"][rows], split["c_doc"], args.deadline)
        del vectors
        data = build_features(split, rows, dense, args.depth)
        splits[name] = data
        print(f"{name} · features for {len(rows)} queries · {time.monotonic()-started:.1f}s", flush=True)
        if name == "Test":
            pcr.atomic_json(hybrid_dir / "prediction.json",
                            pcr.prediction(data["hybrid_top"], split))
        if name == "Dev":
            report = pcr.metrics(data["hybrid_top"], data["gold"])
            pcr.atomic_json(hybrid_dir / "dev.json", report)
            diagnostic = oracle(data)
            pcr.atomic_json(args.root / "shortlist_dev.json", diagnostic)
            print(f"Hybrid Dev F1@7 {report['f1']:.4f} · "
                  f"shortlist recall {diagnostic['shortlist_recall']:.4f} · "
                  f"oracle F1@7 {diagnostic['oracle_micro_f1_at_7']:.4f} · target 0.4500",
                  flush=True)
    del model
    output = args.root / "lambdamart"
    output.mkdir(exist_ok=True)
    ranker, report = fit_ranker(splits["Train"], splits["Dev"], output, args)
    test = splits["Test"]
    test_x, _, _ = flatten(test, training=False)
    preds = ranker.predict(test_x, num_threads=4)
    ranks = rerank(preds, test, report["model_weight"])
    pcr.atomic_json(output / "prediction.json", pcr.prediction(ranks, bundle["Test"]))
    pcr.atomic_json(output / "dev.json", report)
    print(f"LambdaMART Dev F1@7 {report['f1']:.4f} · "
          f"ranker weight {report['model_weight']:.2f} · {report['trees']} trees", flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root", type=Path, required=True)
    parser.add_argument("--checkpoint", type=Path, required=True)
    parser.add_argument("--deadline", type=float, required=True)
    parser.add_argument("--encode-batch", type=int, default=8)
    parser.add_argument("--depth", type=int, default=60)
    parser.add_argument("--seed", type=int, default=781)
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    torch.set_num_threads(1)
    torch.manual_seed(args.seed)
    if torch.cuda.device_count() != 1:
        raise RuntimeError("Expose exactly one CUDA GPU to the v4 worker")
    bundle, checkpoint = pcr.load_inputs(args)
    run(args, bundle, checkpoint, torch.device("cuda:0"))


if __name__ == "__main__":
    main()
'''
(RUN_ROOT / 'pcr_ranker.py').write_text(source, encoding='utf-8')
print('Prepared pcr_ranker.py')

## Run and export

Full runs save the selected prediction.json at the top level and separate hybrid and LambdaMART predictions in their method folders. Smoke outputs are partial and should stay out of the competition. The run saves summary.json, the shortlist oracle report, checkpoints, histories and worker logs.

In [ ]:
def log(message):
    print(f"[{(time.monotonic() - STARTED) / 60:5.1f} min] {message}", flush=True)


def verify_runtime():
    import torch
    if not 0 <= GPU_INDEX < torch.cuda.device_count():
        raise RuntimeError(f"GPU_INDEX={GPU_INDEX} is unavailable. Enable a Kaggle GPU.")
    log(f"v4 device {GPU_INDEX} · {torch.cuda.get_device_name(GPU_INDEX)}")
    if CHECKPOINT.is_file():
        return CHECKPOINT
    matches = [path for path in Path("/kaggle/input").rglob("final_model(2).pt")
               if "phase2v4" in path.parts]
    if len(matches) != 1:
        raise FileNotFoundError("Attach the phase2v4 input or set CHECKPOINT to its final_model(2).pt.")
    return matches[0]


def run_stage(name, jobs, deadline):
    log(name)
    stage_dir = RUN_ROOT / "logs"
    stage_dir.mkdir(exist_ok=True)
    workers = []
    statuses = {}
    try:
        for label, command, gpu, overrides in jobs:
            path = stage_dir / f"{label}.log"
            env = os.environ.copy()
            if gpu is not None:
                env["CUDA_VISIBLE_DEVICES"] = str(gpu)
            env.update(OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", **overrides)
            with path.open("w", encoding="utf-8") as stream:
                process = subprocess.Popen(
                    [sys.executable, "-u", *map(str, command)],
                    cwd=RUN_ROOT, env=env, stdout=stream, stderr=subprocess.STDOUT,
                )
            workers.append((label, process, path.open(encoding="utf-8", errors="replace")))
        active = workers.copy()
        while active and time.monotonic() < deadline:
            for item in active.copy():
                label, process, reader = item
                lines = reader.read().strip()
                if lines:
                    print(lines, flush=True)
                if process.poll() is not None:
                    lines = reader.read().strip()
                    if lines:
                        print(lines, flush=True)
                    statuses[label] = process.returncode
                    active.remove(item)
            if active:
                time.sleep(.5)
    finally:
        for label, process, reader in workers:
            if process.poll() is None:
                process.kill()
                statuses[label] = "deadline_or_interrupt"
            else:
                statuses.setdefault(label, process.returncode)
            lines = reader.read().strip()
            if lines:
                print(lines, flush=True)
            reader.close()
    for label, status in statuses.items():
        if status != 0:
            log(f"{label} status {status} · see {stage_dir / (label + '.log')}")
    return statuses


def worker(action, method, deadline, checkpoint):
    return [
        "pcr_worker.py", action, method, "--root", RUN_ROOT, "--checkpoint", checkpoint,
        "--deadline", deadline, "--query-batch", QUERY_BATCH, "--encode-batch", ENCODE_BATCH,
        "--validation-seconds", VALIDATION_SECONDS, *(["--smoke"] if SMOKE_MODE else [])]


def stage_end(minutes, reserve_minutes):
    return min(time.monotonic() + minutes * 60, DEADLINE - reserve_minutes * 60 - 15)


def run():
    checkpoint = verify_runtime()
    log(f"Phase 2 v4 base checkpoint · {checkpoint}")
    setup_command = ["pcr_prepare.py", "--root", RUN_ROOT, "--length", CHUNK_LENGTH,
                     "--stride", WINDOW_STRIDE, "--windows", MAX_WINDOWS]
    for flag, value in (("--data", LOCAL_DATA_ROOT), ("--tokenizer", LOCAL_TOKENIZER_DIR)):
        if value is not None:
            setup_command.extend((flag, value))

    if SMOKE_MODE:
        setup_command.append("--smoke")
    statuses = run_stage("Preparing token windows and legal lexical features",
                         [("setup", setup_command, None, {})],
                         stage_end(8 if SMOKE_MODE else 15, 28))
    if statuses.get("setup") != 0 or not (RUN_ROOT / "inputs.pt").is_file():
        raise RuntimeError("Setup did not finish. See the setup log above.")

    def trained():
        directory = RUN_ROOT / "lorma"
        report = directory / "dev.json"
        return report.is_file() and (directory / json.loads(report.read_text())["weights_file"]).is_file()

    train_deadline = stage_end(8 if SMOKE_MODE else 43, 20)
    train_job = ("lorma", worker("train", "lorma", train_deadline - 3, checkpoint), GPU_INDEX, {})
    statuses.update(run_stage("LoRMA+ with three-check loss patience", [train_job], train_deadline))

    export_deadline = stage_end(8, 0) if SMOKE_MODE else DEADLINE - 15
    if trained() and time.monotonic() + 20 < export_deadline:
        command = ["pcr_ranker.py", "--root", RUN_ROOT, "--checkpoint", checkpoint,
                   "--deadline", export_deadline - 3, "--encode-batch", ENCODE_BATCH,
                   "--depth", SHORTLIST_DEPTH]
        if SMOKE_MODE:
            command.append("--smoke")
        statuses.update(run_stage("Hybrid shortlist, LambdaMART, and Test export",
                                  [("ranker", command, GPU_INDEX, {})], export_deadline))

    def report(method):
        directory = RUN_ROOT / method
        dev, pred = directory / "dev.json", directory / "prediction.json"
        return json.loads(dev.read_text()) if dev.is_file() and pred.is_file() else None

    hybrid, learned = report("hybrid"), report("lambdamart")
    if hybrid is None and trained() and time.monotonic() + 20 < export_deadline:
        job = ("export_lorma", worker("export", "lorma", export_deadline - 3, checkpoint), GPU_INDEX, {})
        statuses.update(run_stage("Exporting retriever fallback", [job], export_deadline))

    if hybrid is not None:
        selected = "lambdamart" if learned is not None and learned["f1"] > hybrid["f1"] else "hybrid"
    else:
        selected = "lorma" if report("lorma") is not None else "bm25_fallback"
    filename = "smoke_prediction.json" if SMOKE_MODE else "prediction.json"
    path = RUN_ROOT / selected / "prediction.json"
    if selected == "bm25_fallback":
        path = RUN_ROOT / "bm25_prediction.json"
    target = RUN_ROOT / filename
    shutil.copyfile(path, target)
    diagnostic = RUN_ROOT / "shortlist_dev.json"
    training = RUN_ROOT / "lorma" / "training.json"
    summary = dict(
        mode="smoke" if SMOKE_MODE else "full", selected=selected,
        complete=hybrid is not None and learned is not None,
        hybrid=hybrid, lambdamart=learned, worker_statuses=statuses,
        shortlist=json.loads(diagnostic.read_text()) if diagnostic.is_file() else None,
        training=json.loads(training.read_text()) if training.is_file() else None,
        elapsed_seconds=time.monotonic() - STARTED,
    )
    (RUN_ROOT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    if not summary["complete"]:
        log("INCOMPLETE · inspect summary.json and worker logs for fallback details")
    log(f"Selected {selected} · {target}")
    return summary


summary = run()

In [ ]:
UPLOAD_TO_HF = True  # Set True to upload after local export.
HF_REPO = "DumbLad/phase7-v4-pcr"

def upload_run():
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient

    if not (RUN_ROOT / "prediction.json").is_file():
        print("HF upload skipped · run training and export first.")
        return

    token = None
    try:
        secrets = UserSecretsClient()
        for name in ("HF_WRITE", "HF"):
            try:
                token = secrets.get_secret(name)
            except Exception:
                continue
            if token:
                break
    except Exception:
        pass
    if not token:
        print("HF upload skipped · enable HF or HF_WRITE for this Kaggle notebook. Local outputs are saved.")
        return

    try:
        api = HfApi(token=token)
        api.create_repo(HF_REPO, repo_type="model", private=True, exist_ok=True)
        if not api.repo_info(HF_REPO, repo_type="model").private:
            print("HF upload skipped · destination repo must be private. Local outputs are saved.")
            return
        commit = api.upload_folder(
            repo_id=HF_REPO, repo_type="model", folder_path=RUN_ROOT,
            path_in_repo=f"runs/{RUN_ROOT.name}",
            allow_patterns=['prediction.json', 'summary.json', 'hard_negatives.json', 'shortlist_dev.json', '*/lambda_model.txt', 'bm25_prediction.json', '*_dev.json', '*/prediction.json', '*/dev.json', '*/training.json', '*/history.json', '*/checkpoint_*.pt', 'logs/*.log', '*.py'],
            commit_message=f"Upload Phase 7 v4 run {RUN_ROOT.name}",
        )
    except Exception as error:
        print(f"HF upload failed ({type(error).__name__}) · local outputs are saved. "
              "Check token write permissions and Internet, then retry this cell.")
        return
    print(f"Uploaded: {commit.commit_url}")

if UPLOAD_TO_HF and not SMOKE_MODE:
    upload_run()
